In [7]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score

from xgboost import XGBClassifier

RANDOM_STATE = 42


In [8]:
# Adjust path as needed
df = pd.read_csv("datasets/dataset1_text_rich_transactions_harder.csv")

# Keep only the columns needed for this model
# description: feature; category_label: target
df = df.dropna(subset=["description", "category_label"]).copy()

print("Rows after dropna:", len(df))
print("Sample columns:", df.columns.tolist())


Rows after dropna: 4899
Sample columns: ['transaction_id', 'transaction_date', 'amount', 'currency', 'description', 'vendor_name', 'gst_applicable', 'gst_slab', 'itc_eligible', 'category_label', 'is_anomaly']


In [9]:
def clean_description(s: str) -> str:
    """Deterministic cleaning, no vendor extraction."""
    if not isinstance(s, str):
        return ""
    s = s.lower()

    # normalize common banking/payment tokens
    replacements = {
        r"\bupi\b": " upi ",
        r"\bimps\b": " imps ",
        r"\bneft\b": " neft ",
        r"\brtgs\b": " rtgs ",
        r"\bcard\b": " card ",
        r"\bauto[- ]?debit\b": " autodebit ",
        r"\bbill\b": " bill ",
        r"\binv\b": " invoice ",
        r"\btaxinv\b": " taxinvoice ",
        r"\brcpt\b": " receipt ",
    }
    for pat, repl in replacements.items():
        s = re.sub(pat, repl, s)

    # remove noisy IDs, ref numbers, invoice IDs etc.
    noise_tokens = [
        r"fy\d{2}",          # fy24, fy25
        r"q[1-4]",          # q1, q2
        r"ref\s*\d+",       # ref 1234
        r"inv/?\d+",        # inv/2404/001
        r"bill/?\d+",
        r"rcpt/?\d+",
        r"taxinv/?\d+",
        r"\d{2}-\d{2}-\d{4}",  # dates like 01-04-2024
    ]
    for pat in noise_tokens:
        s = re.sub(pat, " ", s)

    # keep alphanumerics and spaces
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["description_clean"] = df["description"].apply(clean_description)

print("Example cleaned descriptions:")
print(df["description"].head(3).tolist())
print(df["description_clean"].head(3).tolist())


Example cleaned descriptions:
['imhs ikea rcpt/2404/001 subs - exp', ' upi awfis rcpt/2404/002 fy24 - rent exp# ref 6160', 'ZOMATO']
['imhs ikea receipt 2404 001 subs exp', 'upi awfis receipt 2404 002 rent exp', 'zomato']


In [11]:
TEXTCOL = "description_clean"
TARGETCOL = "category_label"

X_text = df[TEXTCOL].astype(str)
y = df[TARGETCOL].astype(str)

# Label encode target
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Train/validation split
X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_text,
    y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=RANDOM_STATE,
)

print("Train size:", len(X_train_text), "Val size:", len(X_val_text))

# TF-IDF: description-only
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),  # uni + bi-grams
    min_df=3,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)

print("TF-IDF shapes:", X_train_tfidf.shape, X_val_tfidf.shape)


Train size: 3919 Val size: 980
TF-IDF shapes: (3919, 1795) (980, 1795)


In [12]:
xgb_desc_only = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    max_depth=7,        # slightly conservative vs rich-text model
    learning_rate=0.1,
    n_estimators=250,   # 250-300; adjust if needed
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

xgb_desc_only.fit(X_train_tfidf, y_train)

# Validation predictions
y_val_proba = xgb_desc_only.predict_proba(X_val_tfidf)
y_val_pred = y_val_proba.argmax(axis=1)

y_val_true_str = le.inverse_transform(y_val)
y_val_pred_str = le.inverse_transform(y_val_pred)

print("Description-only TF-IDF + XGBoost")
print(classification_report(y_val_true_str, y_val_pred_str))
print("Macro F1:", f1_score(y_val_true_str, y_val_pred_str, average="macro"))


Description-only TF-IDF + XGBoost
                 precision    recall  f1-score   support

Exempt Services       0.65      0.65      0.65        26
    IT Services       0.95      0.96      0.95       216
          Meals       0.97      0.95      0.96        74
Office Supplies       0.91      0.95      0.93       124
           Rent       0.94      0.94      0.94       151
       Software       0.97      0.97      0.97       115
       Training       1.00      0.94      0.97        49
         Travel       0.96      0.95      0.96        82
      Utilities       0.98      0.96      0.97       143

       accuracy                           0.95       980
      macro avg       0.93      0.92      0.92       980
   weighted avg       0.95      0.95      0.95       980

Macro F1: 0.922108213047192


In [13]:
def predict_description_only(description: str):
    """
    Inference helper for a single transaction description.
    Output format matches orchestrator expectations.
    """
    desc_clean = clean_description(description)
    X_tfidf = tfidf.transform([desc_clean])

    proba = xgb_desc_only.predict_proba(X_tfidf)[0]
    pred_idx = int(np.argmax(proba))
    pred_label = le.inverse_transform([pred_idx])[0]
    max_proba = float(np.max(proba))

    return {
        "predicted_category": pred_label,
        "prediction_probability": max_proba,
        "confidence_score": max_proba,
        "needs_review": False,  # orchestrator will overwrite using threshold
    }

# Quick sanity check
example_desc = df["description"].iloc[0]
print(example_desc)
print(predict_description_only(example_desc))


imhs ikea rcpt/2404/001 subs - exp
{'predicted_category': 'Office Supplies', 'prediction_probability': 0.9944949746131897, 'confidence_score': 0.9944949746131897, 'needs_review': False}


In [14]:
def predict_batch_descriptions(descriptions):
    """
    descriptions: list of raw description strings
    returns: list of dicts with prediction fields
    """
    cleaned = [clean_description(d) for d in descriptions]
    X_tfidf = tfidf.transform(cleaned)

    proba_all = xgb_desc_only.predict_proba(X_tfidf)
    pred_idx_all = proba_all.argmax(axis=1)
    pred_labels = le.inverse_transform(pred_idx_all)

    outputs = []
    for desc, label, proba_vec in zip(descriptions, pred_labels, proba_all):
        max_proba = float(np.max(proba_vec))
        outputs.append({
            "predicted_category": label,
            "prediction_probability": max_proba,
            "confidence_score": max_proba,
            "needs_review": False,  # set later by orchestrator
        })
    return outputs

# Example:
sample_descs = df["description"].head(5).tolist()
batch_out = predict_batch_descriptions(sample_descs)
for o in batch_out:
    print(o)


{'predicted_category': 'Office Supplies', 'prediction_probability': 0.9944949746131897, 'confidence_score': 0.9944949746131897, 'needs_review': False}
{'predicted_category': 'Rent', 'prediction_probability': 0.9975040555000305, 'confidence_score': 0.9975040555000305, 'needs_review': False}
{'predicted_category': 'Meals', 'prediction_probability': 0.963996171951294, 'confidence_score': 0.963996171951294, 'needs_review': False}
{'predicted_category': 'Office Supplies', 'prediction_probability': 0.9986573457717896, 'confidence_score': 0.9986573457717896, 'needs_review': False}
{'predicted_category': 'Utilities', 'prediction_probability': 0.9958227872848511, 'confidence_score': 0.9958227872848511, 'needs_review': False}


In [15]:
# # Version tag for CI/CD compatibility
# MODEL_VERSION = "v1"

# joblib.dump(tfidf, f"artifacts/description_only_vectorizer_{MODEL_VERSION}.pkl")
# joblib.dump(xgb_desc_only, f"artifacts/description_only_xgb_model_{MODEL_VERSION}.pkl")
# joblib.dump(le, f"artifacts/description_only_label_encoder_{MODEL_VERSION}.pkl")

# print("Saved artifacts for description-only model:", MODEL_VERSION)
